### Tools ->  model can request to call tools to perform task such as database, websearch

In [1]:
import os
from dotenv import load_dotenv
# Load environment variables from .env file

load_dotenv()
from langchain.chat_models import init_chat_model


In [2]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") #Sets an environment variable inside the current Python process.
model = init_chat_model("qwen/qwen3-32b", model_provider="groq", api_key=os.environ["GROQ_API_KEY"])
response = model.invoke("What is RAG?")
response

AIMessage(content='<think>\nOkay, the user asked What is RAG? I need to first confirm what RAG is. RAG is the abbreviation of Retrieval-Augmented Generation. This is a technology that combines retrieval and generation to enhance the capabilities of large models.\n\nFirst, I should briefly explain the basic concept of RAG, including how it integrates retrieval and generation. Then elaborate on its working mechanism: the retrieval phase is to find relevant documents in the database based on the user\'s question, and the generation phase is to use the information provided by the retrieval to generate answers.\n\nNext, I need to mention the advantages of RAG, such as improving the accuracy and reliability of answers, enabling the model to respond to the latest information and reduce hallucinations. At the same time, it may also be mentioned that RAG has some challenges, such as the need for a good retrieval system and the problem of information overload.\n\nIt is also important to note tha


### Create tools
​
Basic tool definition
The simplest way to create a tool is with the @tool decorator. By default, the function’s docstring becomes the tool’s description that helps the model understand when to use it:

In [9]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


tools = [get_weather]

weather_agent = create_agent(
    model=init_chat_model(
        "qwen/qwen3-32b",
        model_provider="groq",
        api_key=os.environ["GROQ_API_KEY"]
    ),
    tools=tools
)

for tool in tools:
    print(f"Tool name: {tool.name}")
    print(f"Description: {tool.description}")


Tool name: get_weather
Description: Get weather for a given city.


#### Tools call

In [11]:
response = weather_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in New York?"}]}
)
response


{'messages': [HumanMessage(content="What's the weather in New York?", additional_kwargs={}, response_metadata={}, id='ab4fc2f6-f83c-4bea-b2ad-c4fb3b65ce01'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. The required parameter is city, and it\'s a string. Since the user specified New York, I need to call get_weather with "New York" as the city argument. I\'ll make sure the JSON is correctly formatted with the city name in quotes. No other parameters are needed. Let me structure the tool_call accordingly.\n', 'tool_calls': [{'id': '0qtzxzcfy', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 154, 'total_tokens': 277, 'completion_time': 0.18637069, 'completion_tokens_details': {'reasoning_toke

## Breakdown Agent responce

In [ ]:
# Display the response messages through a loop to handle different message types

for msg in response["messages"]:   
    print("\n" + "="*50)

    if msg.type == "human":
        print("👤 USER:")
        print(msg.content)

    elif msg.type == "ai":
        print("🤖 AI:")
        print(msg.content)

        if msg.tool_calls:
            print("\n🔧 TOOL CALLS:")
            for call in msg.tool_calls:
                print(f"Tool: {call['name']}")
                print(f"Args: {call['args']}")

    elif msg.type == "tool":
        print("🛠 TOOL RESPONSE:")
        print(msg.content)


👤 USER:
What's the weather in New York?

🤖 AI:


🔧 TOOL CALLS:
Tool: get_weather
Args: {'city': 'New York'}

🛠 TOOL RESPONSE:
It's always sunny in New York!

🤖 AI:
The weather in New York is always sunny! ☀️
